# dsh 离线 bundle 构建(联网 CPU kernel)

产物 dsh-bundle.tgz 供提交 kernel 断网解包使用。

In [ ]:
import subprocess, os
from pathlib import Path
W = Path("/kaggle/working")
os.chdir(W)

def run(cmd, **kw):
    print("+", cmd, flush=True)
    subprocess.check_call(cmd, shell=True, **kw)

NODE = "node-v22.19.0-linux-x64"
run(f"wget -q https://nodejs.org/dist/v22.19.0/{NODE}.tar.xz")
run(f"tar xf {NODE}.tar.xz && rm {NODE}.tar.xz && mv {NODE} node")
os.environ["PATH"] = f"{W}/node/bin:" + os.environ["PATH"]
run("node --version")
run("npm install -g --force pnpm@11.7.0")  # 仓库 packageManager 同款; 不走 corepack
run("pnpm --version")

In [ ]:
run("git clone --depth 1 -b fix/tool-call-empty-string-deltas "
    "https://github.com/jinbowang1/deepseek-harness dsh-src")
os.chdir(W / "dsh-src")
env = dict(os.environ, DSH_LEFTHOOK_ALLOW_HOOKS_PATH_OVERRIDE="1", CI="1")
subprocess.check_call("pnpm install --frozen-lockfile", shell=True, env=env)
subprocess.check_call("pnpm run build", shell=True, env=env)
assert (W / "dsh-src/apps/cli/lib/bin.js").exists(), "build 没产出 apps/cli/lib/bin.js"
print("build done", flush=True)

In [ ]:
os.chdir(W)
# 剔除 .git 和平台无关的大缓存; 保留 node_modules 符号链接结构(tar 原生支持)
run("rm -rf dsh-src/.git")
run("tar czf dsh-bundle.tgz node dsh-src")
run("rm -rf node dsh-src")
run("ls -lh dsh-bundle.tgz")
# 冒烟: 解包后 bin.js 能 --version (在本 kernel 内验证包完整)
run("mkdir -p /tmp/smoke && tar xzf dsh-bundle.tgz -C /tmp/smoke")
run("DSH_HOME=/tmp/smoke/home /tmp/smoke/node/bin/node "
    "/tmp/smoke/dsh-src/apps/cli/lib/bin.js --version")
print("bundle 冒烟通过", flush=True)